In [11]:
import os
import random
import math
from PIL import Image, ImageDraw

def generate_simple_polygon(n_vertices, r_min=0.5, r_max=1.5, tol=1e-6, max_attempts=1000):
    """
    Generate a simple (non-self-intersecting) closed polygon with unique edge lengths.
    """
    if n_vertices < 3:
        raise ValueError("Number of vertices must be at least 3")

    for attempt in range(max_attempts):
        angles = sorted(random.uniform(0, 2 * math.pi) for _ in range(n_vertices))
        radii  = [random.uniform(r_min, r_max) for _ in range(n_vertices)]
        points = [(radii[i] * math.cos(angles[i]),
                   radii[i] * math.sin(angles[i]))
                  for i in range(n_vertices)]

        # compute edge lengths
        lengths = []
        for i in range(n_vertices):
            x1, y1 = points[i]
            x2, y2 = points[(i + 1) % n_vertices]
            lengths.append(math.hypot(x2 - x1, y2 - y1))

        sorted_lens = sorted(lengths)
        if all(abs(a - b) > tol for a, b in zip(sorted_lens, sorted_lens[1:])):
            return points

    raise RuntimeError(f"Failed to generate a valid polygon after {max_attempts} attempts.")


def save_polygon_png(points, filename,
                     canvas_size=300, shape_size=200,
                     fill='#cce5ff', outline='#337ab7', outline_width=2,
                     render_scale=4):
    """
    Draw the polygon at `render_scale`× resolution, then downsample to
    canvas_size×canvas_size with Lanczos for anti-aliasing.
    """
    # scaled sizes
    CS = canvas_size * render_scale
    SS = shape_size   * render_scale
    OW = outline_width * render_scale

    # 原始坐标 bounds
    xs = [p[0] for p in points]
    ys = [p[1] for p in points]
    min_x, max_x = min(xs), max(xs)
    min_y, max_y = min(ys), max(ys)
    width_x = max_x - min_x
    width_y = max_y - min_y

    # 缩放因子（高分辨率下）
    scale = SS / max(width_x, width_y)
    cx = cy = CS / 2

    # map to high-res canvas
    mapped = []
    for x, y in points:
        px = (x - (min_x + max_x) / 2) * scale + cx
        py = -(y - (min_y + max_y) / 2) * scale + cy
        mapped.append((px, py))

    # high-res 绘制白底
    im_hr = Image.new('RGB', (CS, CS), 'white')
    draw = ImageDraw.Draw(im_hr)
    draw.polygon(mapped, fill=fill, outline=outline)
    if OW > 1:
        # 更粗 outline
        for i in range(len(mapped)):
            x1, y1 = mapped[i]
            x2, y2 = mapped[(i+1) % len(mapped)]
            draw.line((x1, y1, x2, y2), fill=outline, width=OW)

    # downsample to target size
    im = im_hr.resize((canvas_size, canvas_size), resample=Image.LANCZOS)
    im.save(filename, format='PNG')


if __name__ == "__main__":
    try:
        count = int(input("Enter the number of polygons to generate: "))
        if count <= 0:
            raise ValueError
    except ValueError:
        print("Please enter a positive integer.")
        exit(1)

    output_dir = "polygons"
    os.makedirs(output_dir, exist_ok=True)

    # 可以根据需要调整这些参数
    n_vertices   = 8
    r_min, r_max = 0.5, 1.5
    tol          = 1e-6
    max_attempts = 1000

    # 这里把 render_scale 调大可以获得更细腻的边缘，但绘制速度会略慢
    canvas_size  = 300
    shape_size   = 200
    render_scale = 4
    outline_w    = 2

    for i in range(count):
        pts = generate_simple_polygon(n_vertices, r_min, r_max, tol, max_attempts)
        out_path = os.path.join(output_dir, f"{i}.png")
        save_polygon_png(
            pts, out_path,
            canvas_size=canvas_size,
            shape_size=shape_size,
            fill='#cce5ff',
            outline='#337ab7',
            outline_width=outline_w,
            render_scale=render_scale
        )
        print(f"[{i+1}/{count}] Saved: {out_path}")

    print(f"Generated {count} high-quality polygons in '{output_dir}'.")


[1/10] Saved: polygons\0.png
[2/10] Saved: polygons\1.png
[3/10] Saved: polygons\2.png
[4/10] Saved: polygons\3.png
[5/10] Saved: polygons\4.png
[6/10] Saved: polygons\5.png
[7/10] Saved: polygons\6.png
[8/10] Saved: polygons\7.png
[9/10] Saved: polygons\8.png
[10/10] Saved: polygons\9.png
Generated 10 high-quality polygons in 'polygons'.


In [13]:
import os
import random
from PIL import Image, ImageOps

def augment_polygons(input_dir='polygons',
                     sub_count=5,
                     ans_file='ans.txt',
                     rotation_resample=Image.BICUBIC):
    """
    Read original polygon PNG files from `input_dir` (filenames without '-'),
    generate `sub_count` augmented images for each, named as base-1.png ... base-sub_count.png.
    The images are either just rotated (record 'yes') or mirrored then rotated (record 'no').
    Transparent areas after rotation are filled white.
    Uses BICUBIC resampling on rotate to reduce锯齿.
    Finally writes an `ans_file` listing:
        <sub-image filename> <space> yes|no
    """
    # Collect originals (no '-' in the stem)
    files = sorted(
        f for f in os.listdir(input_dir)
        if f.lower().endswith('.png') and '-' not in os.path.splitext(f)[0]
    )
    if not files:
        print(f"No original PNG files found in '{input_dir}'.")
        return

    ans_lines = []

    for fname in files:
        base, _ = os.path.splitext(fname)
        img_path = os.path.join(input_dir, fname)
        try:
            img = Image.open(img_path).convert('RGBA')
        except Exception as e:
            print(f"Cannot open {img_path}: {e}")
            continue

        for k in range(1, sub_count + 1):
            direct_rotate = random.choice([True, False])
            angle = random.uniform(0, 360)

            aug = img
            if not direct_rotate:
                aug = ImageOps.mirror(aug)

            # 抗锯齿旋转
            aug = aug.rotate(angle, expand=True, resample=rotation_resample)

            # 白底合成
            bg = Image.new('RGBA', aug.size, (255, 255, 255, 255))
            bg.paste(aug, (0, 0), mask=aug)

            final = bg.convert('RGB')
            out_name = f"{base}-{k}.png"
            out_path = os.path.join(input_dir, out_name)
            final.save(out_path, format='PNG')

            tag = 'yes' if direct_rotate else 'no'
            ans_lines.append(f"{out_name} {tag}")

        img.close()

    # 写 ans.txt
    with open(ans_file, 'w') as f:
        f.write('\n'.join(ans_lines))

    print(f"Generated {len(ans_lines)} sub-images, recorded in '{ans_file}'.")


if __name__ == "__main__":
    # 调用示例
    augment_polygons(
        input_dir='polygons',
        sub_count=5,
        ans_file='ans.txt',
        rotation_resample=Image.BICUBIC
    )


Generated 50 sub-images, recorded in 'ans.txt'.
